# Decomposing a fault: cuts, the reduction factor and a location without its electrode

A 20 kV cable feeder with six substations, faulted in the middle. The point
of this example is not the fault solve itself — that is the
[minimal example](../minimal/) — but the analysis layer on top of it:

1. the **reduction factor** and how it is checked a second way,
2. **`analyze_cuts`** — what each direction away from the fault contributes,
3. **`bus_response`** — the whole network response to *any* electrode at the
   faulted station, from two solves,
4. **`run_sweep` + `summarize`/`classify`** — the same study across a range of
   soil resistivities,
5. **`run_reference_cases`** — the closed-form checks the package runs against
   itself.

In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt
import polars as pl

import groundinsight as gi

pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_width_chars(120)

## 1. The network

Six substations `S0 … S5`, connected by 1.2 km sections of a 20 kV cable whose
shield-earth loop carries 0.25 Ω/km and 1.9 mH/km, with 0.05 Ω/km and
1.85 mH/km of Carson earth-return mutual coupling to the phase conductor.

`phase_impedance_formula` describes the **phase** conductor — the faulted one
whose current induces the longitudinal EMF on the shield. It never enters the
nodal admittance matrix; it is what the automatic phase-current distribution
solves on. On a radial feeder like this one the phase current is unambiguous
either way, but declaring it costs nothing and makes the model honest if a ring
is closed later.

In [ ]:
TWO_PI = 2 * 3.141592653589793
FREQS = [50.0]
N_BUSES = 6
SECTION_KM = 1.2
I_FAULT = 3000.0          # A, 3*I_0 at the infeed

net = gi.create_network(name="DecompositionDemo", frequencies=FREQS)

station = gi.BusType(
    name="Station",
    system_type="Substation",
    voltage_level=20,
    impedance_formula="rho * 0.02",
)
cable = gi.BranchType(
    name="NA2XS2Y-1x300",
    grounding_conductor=True,
    self_impedance_formula=f"(0.25 + j * f * 1.9e-3 * {TWO_PI}) * l",
    mutual_impedance_formula=f"(0.05 + j * f * 1.85e-3 * {TWO_PI}) * l",
    phase_impedance_formula=f"(0.10 + j * f * 0.4e-3 * {TWO_PI}) * l",
)

for i in range(N_BUSES):
    gi.create_bus(name=f"S{i}", type=station, network=net)
for i in range(N_BUSES - 1):
    gi.create_branch(
        name=f"C{i}{i + 1}", type=cable,
        from_bus=f"S{i}", to_bus=f"S{i + 1}",
        length=SECTION_KM, network=net,
    )

gi.create_source(name="infeed", bus="S0", values={50.0: I_FAULT}, network=net)
gi.create_fault(name="mid", bus="S3", scalings={50.0: 1.0}, network=net)

gi.run_fault(network=net, fault="mid")
net.res_all_impedances()

## 2. The reduction factor

`reduction_factor` (field `value`) is $r = |I_E| / |3 I_0|$, the quantity
EN 50522 means. The earth current $I_E = U_E / Z_E$ is, by Norton's theorem,
the current an ideal electrode at the fault bus would take: the part of the
fault current that does not return to the infeed through the shields.

The earthing voltage and the earthing impedance of the norm are taken **at the
fault bus**: $U_E$ is `u_earthing`, the EPR of the fault bus itself, and $Z_E$
is `z_earthing`, the driving-point impedance of the earthing network there —
the reported grounding impedance. The norm's chain
$U_E = r \cdot 3 I_0 \cdot Z_E$ therefore holds exactly, on this short feeder
as well; it checks nothing. The cell checks $r$ a second way instead: it earths
S3 ideally and reads the current into that electrode.

In [ ]:
rf = net.results["mid"].reduction_factor

z_e = abs(complex(rf.z_earthing[50.0]))
u_e = abs(complex(rf.u_earthing[50.0]))
epr_fault_bus = next(b for b in net.results["mid"].buses if b.name == "S3").uepr

print(f"reduction_factor  r                  = {rf.value[50.0]:.6f}")
print(f"I_E = U_E / Z_E                      = {abs(complex(rf.i_earth[50.0])):.1f} A")
print(f"U_E  (u_earthing = EPR at S3)        = {u_e:.2f} V")
print(f"Z_E  (z_earthing = driving point S3) = {z_e:.4f} ohm")
print(f"r * 3*I_0 * Z_E                      = {rf.value[50.0] * I_FAULT * z_e:.2f} V")

# Second way: S3 earthed ideally, the current into that electrode over 3*I_0.
ideal = net.model_copy(deep=True)
ideal.buses["S3"].impedance = {50.0: gi.ComplexNumber(real=1e-9, imag=0.0)}
gi.run_fault(ideal, fault="mid")
u_ideal = next(complex(b.uepr_freq[50.0]) for b in ideal.results["mid"].buses if b.name == "S3")
print(f"r from an ideal electrode at S3      = {abs(u_ideal / 1e-9) / I_FAULT:.6f}")

# z_earthing is the same number the impedance frame reports as Z_E ...
assert abs(z_e - abs(complex(net.results["mid"].grounding_impedance.value[50.0]))) < 1e-12
# ... and U_E is the fault bus's own EPR (one frequency, so the RMS is |u_F|).
assert abs(u_e - epr_fault_bus) < 1e-9 * u_e
assert abs(abs(u_ideal / 1e-9) / I_FAULT - rf.value[50.0]) < 1e-6 * rf.value[50.0]

## 3. Splitting the network at the fault

A `Cut` is a named set of branches, all incident to the fault bus. The analysis
reports what each direction contributes:

    parallel impedance left --- fault location --- parallel impedance right

`Z_side` comes from **source-free current division**: one ampere is injected at
the fault bus with all sources and mutual injections removed, and the share
leaving through each cut gives $Z_{side} = u_{fault} / i_{cut}$. The
decomposition closes by construction — $1/Z_{dp} = 1/Z_{local} +
\sum 1/Z_{side}$: the side admittances and the local one add up to the
driving-point admittance of the whole network.

Note `sides_are_disjoint`: only when the directions do not reach around into
one another do the per-side current quantities have an unambiguous meaning.

In [ ]:
towards_infeed = gi.Cut(
    name="towards_infeed", branches=["C23"],
    description="S3 -> S2 -> S1 -> S0, the direction the source feeds from",
)
towards_end = gi.Cut(
    name="towards_open_end", branches=["C34"],
    description="S3 -> S4 -> S5, the passive tail",
)

cuts = gi.analyze_cuts(net, fault="mid", cuts=[towards_infeed, towards_end])
print("sides_are_disjoint:", cuts.sides_are_disjoint)

frame = cuts.to_polars()
frame.select([
    "cut", "side_buses", "Z_side_Ohm", "I_shield_A", "current_share",
    "I_earth_A", "r_side", "kcl_residual",
])

The passive tail gets `r_side = None` rather than a division by zero: no source
current enters from that direction, so there is no share to take a ratio of.
`kcl_residual` is the check that the currents at the fault bus actually close.

## 4. Characterising the location, not the electrode

Adding an electrode is a rank-one change to the nodal matrix, so by
Sherman–Morrison every nodal voltage is a Möbius function of its admittance.
**Two solves with the electrode removed therefore determine the response for
every electrode.** `BusResponse.evaluate(z)` and `.sweep([...])` then cost no
solve at all.

`z_network` is the site-independent number the analysis is built around: the
driving-point impedance of everything *except* the local electrode. It is a
property of the location, not of what happens to be installed there.

In [ ]:
response = gi.bus_response(net, fault="mid", bus="S3")
z_net = response.z_network[50.0]
print(f"Z_net at S3 = {z_net.real:.4f} {z_net.imag:+.4f}j ohm  (|Z| = {abs(z_net):.4f})")

response.extremes().select([
    "case", "Z_bus_Ohm", "Z_driving_point_Ohm", "EPR_V", "EPR_S0_V", "r",
])

The three cases are the extremes **of the local quantities**: `open` (no
electrode at all), `ideal` ($Z_B = 0$) and `worst_passive`, which maximises
$|Z_{dp}|$ at the reactive resonance $Y_B = -j\,\mathrm{Im}(1/Z_{net})$. Both
endpoints are exact limits rather than numerical stand-ins — the ideal electrode
in particular, which the solver rejects outright because a zero impedance is not
invertible.

Read them as a bracket for `EPR_V` and `Z_driving_point_Ohm`, and **not** for
the transfer quantities in the same table. `EPR_S0_V` is not bounded by these
three cases: a purely capacitive electrode at S3 — passive, so inside the same
class — drives the EPR at S0 above the `ideal` value. `r` is the same in all
three rows: it is the current into an ideal electrode at the fault bus, so the
electrode there does not enter it. If a study
needs the worst case of a transfer quantity, sweep for it with `.sweep([...])`,
which costs no solve, rather than reading it off `extremes()`.

What the three cases do surface is the direction of the effect, which is easy to
get backwards. An **ideal** electrode at the faulted station drives the potential
rise at the infeed station *up*, not down: grounding the fault location harder
pulls more current through the shields towards it. Over $|Z_B|$ from
$10^{-3}$ to $10^{4}\,\Omega$ the EPR at S0 falls monotonically, so the plot
below is the whole story for that pair.

In [ ]:
import numpy as np

z_values = np.logspace(-2, 3, 200)
swept = response.sweep([complex(z, 0.0) for z in z_values])
epr_s3 = swept.filter(pl.col("frequency_Hz") == 50.0)["EPR_V"].to_numpy()
epr_s0 = swept.filter(pl.col("frequency_Hz") == 50.0)["EPR_S0_V"].to_numpy()

fig, ax = plt.subplots(figsize=(7.5, 4.2))
ax.semilogx(z_values, epr_s3, label="EPR at S3 (fault bus)")
ax.semilogx(z_values, epr_s0, label="EPR at S0 (infeed)")
ax.set_xlabel(r"electrode impedance at S3, $|Z_B|$  [$\Omega$]")
ax.set_ylabel("EPR [V]")
ax.set_title("Every electrode, from two solves")
ax.grid(True, which="both", alpha=0.3)
ax.legend()
fig.tight_layout()
plt.show()

## 5. The same study across soil resistivities

`run_sweep` solves the fault once per parameter point and stacks the results
into long-format frames that carry the parameters as ordinary columns. Anything
a point overrides is undone in a `finally` block, so a failing point leaves
neither the network nor the remaining points contaminated.

Here the soil resistivity of all six stations moves together, and with it the
split of the return current between the shields and the soil.

In [ ]:
RHOS = [10.0, 30.0, 100.0, 300.0, 1000.0, 3000.0]

points = [
    gi.SweepPoint(
        label=f"rho={rho:g}",
        bus_rho={f"S{i}": rho for i in range(N_BUSES)},
        parameters={"rho": rho},
    )
    for rho in RHOS
]

sweep = gi.run_sweep(net, fault="mid", points=points, cuts=[towards_infeed, towards_end])
impedances = sweep.impedances()
impedances.select([
    "rho", "grounding_impedance_Ohm", "reduction_factor",
])

Over $\rho$ from 10 to 3000 Ω·m the earthing impedance grows by a factor of 66
(0.154 to 10.2 ohm), and the reduction factor falls from 0.31 to 0.023: the
higher the electrode impedances against the shields, the more of the fault
current returns to the infeed through the shields and the less goes into the
soil. At low $\rho$ it reaches the cable's own value
$|1 - Z_m/Z_s| = 0.310003$ — with stations of 0.2 Ω the shields barely matter —
and passes it slightly (0.318 at 30 Ω·m): the factor is a ratio of complex
currents, and the phase of the station electrodes against the shields enters
it. Every branch here has the same $Z_m/Z_s$, so the coupling scales every
potential by $(1 - Z_m/Z_s)$, and $r$ is $|1 - Z_m/Z_s|$ times the share the
galvanic paths alone send into the soil.

In [ ]:
rho = impedances["rho"].to_numpy()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.2))

ax1.semilogx(rho, impedances["reduction_factor"], "o-", label="reduction_factor r")
ax1.set_xlabel(r"soil resistivity $\rho$ [$\Omega$m]")
ax1.set_ylabel("reduction factor [-]")
ax1.set_title("The reduction factor over rho")
ax1.grid(True, which="both", alpha=0.3)
ax1.legend()

ax2.loglog(rho, impedances["grounding_impedance_Ohm"], "d-", color="tab:red")
ax2.set_xlabel(r"soil resistivity $\rho$ [$\Omega$m]")
ax2.set_ylabel(r"$Z_E$ [$\Omega$]")
ax2.set_title("...while the earthing impedance grows 66-fold")
ax2.grid(True, which="both", alpha=0.3)

fig.tight_layout()
plt.show()

`summarize` reduces any numeric column to count, spread, named quantiles and
extremes, grouped or not. `classify` bins a quantity into **user-supplied**
bands — there is deliberately no built-in table of admissible values, because
touch-voltage limits depend on the clearing time, the standard edition and the
assumed additional resistances. The edges come from the caller, who can cite
them.

In [ ]:
# sweep.buses() carries one row per frequency *and* an "RMS" row per bus.
# Pooling them would be meaningless as soon as there is more than one
# frequency, so select the band explicitly before summarising.
per_bus = sweep.buses().filter(pl.col("frequency_Hz") == 50.0)

per_rho = gi.summarize(per_bus, "EPR_V", by=["rho"])
print(per_rho.select(["rho", "n", "mean", "p50", "max"]))

# Edges chosen for this illustration only - substitute the limits your case
# actually has to meet, with the clearing time they belong to.
banded = gi.classify(
    impedances, "grounding_impedance_Ohm",
    edges=[1.0, 5.0], labels=["low", "medium", "high"],
)
banded.select(["rho", "grounding_impedance_Ohm", "class"])

## 6. The closed-form reference cases

Six configurations whose answer is known in closed form, run through the
ordinary public API and compared. Each case states the boundary conditions its
closed form needs — a deviation more often means a condition was not met than
that the model is wrong. These also run as tests, so a change that quietly
breaks agreement cannot reach a release.

In [ ]:
gi.run_reference_cases().select([
    "case", "quantity", "closed_form", "model", "rel_deviation", "agrees",
])

## Where to go next

- [Reduction factors and earthing impedance](../../explanation/reduction-factors/)
  and [Splitting the network, location response](../../explanation/decomposition/)
  carry the derivations: the reduction factor, the cuts,
  and characterising a location without its electrode.
- [Fault-location decomposition](../../api/decomposition/),
  [Location response](../../api/response/),
  [Parameter sweeps](../../api/sweep/),
  [Statistics and classification](../../api/statistics/),
  [Reference cases](../../api/reference/) — the API surface used above.
- [Electrode at the fault location](../cuts_and_rho_f_sweep/) and
  [Location response bracket](../bus_response_extremes/) take the cuts and the
  location response further, with every number checked against a nodal solve.